# Lightweight Face Anti-Spoofing using Frequency Domain Analysis

## Research Overview

This research aims to develop a lightweight face anti-spoofing system using **frequency-domain analysis**.

Face authentication systems can be vulnerable to presentation attacks such as:

- Printed photo attacks
- Smartphone or screen replay attacks
- Other spoofing attempts

Many existing anti-spoofing methods use heavy CNN or Transformer models, which require high computational resources.

Our approach focuses on a lightweight pipeline using **FFT-based frequency analysis** and simple machine-learning classifiers.


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 4:
        indent = "    " * level
        print(indent + os.path.basename(root) + "/")

In [ ]:
import os
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD"

splits = {
    "Training": "LCC_FASD_training",
    "Development": "LCC_FASD_development",
    "Evaluation": "LCC_FASD_evaluation"
}

image_extensions = (".png", ".jpg", ".jpeg", ".bmp")

data = []

for split_name, split_folder in splits.items():
    
    for label in ["real", "spoof"]:
        
        folder_path = os.path.join(
            BASE_PATH,
            split_folder,
            label
        )
        
        count = len([
            file for file in os.listdir(folder_path)
            if file.lower().endswith(image_extensions)
        ])
        
        data.append({
            "Split": split_name,
            "Class": label.capitalize(),
            "Images": count
        })

df_counts = pd.DataFrame(data)

display(df_counts)

print("\nTotal Images:", df_counts["Images"].sum())

In [ ]:
# Step 3: Visualize class distribution
# This graph compares the number of Real and Spoof images
# across Training, Development, and Evaluation datasets.

import matplotlib.pyplot as plt

pivot_df = df_counts.pivot(
    index="Split",
    columns="Class",
    values="Images"
)

pivot_df.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Real vs Spoof Image Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.legend(title="Class")
plt.tight_layout()

plt.show()

# Observation:
# The dataset is highly imbalanced because Spoof images
# are much higher in number than Real images.

In [ ]:
# Step 4: Display sample Real and Spoof images
# This step visually checks examples from both classes
# before starting preprocessing.

import os
import cv2
import matplotlib.pyplot as plt

real_folder = os.path.join(
    BASE_PATH,
    "LCC_FASD_training",
    "real"
)

spoof_folder = os.path.join(
    BASE_PATH,
    "LCC_FASD_training",
    "spoof"
)

real_image_name = os.listdir(real_folder)[0]
spoof_image_name = os.listdir(spoof_folder)[0]

real_image_path = os.path.join(real_folder, real_image_name)
spoof_image_path = os.path.join(spoof_folder, spoof_image_name)

real_img = cv2.imread(real_image_path)
spoof_img = cv2.imread(spoof_image_path)

real_img_rgb = cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)
spoof_img_rgb = cv2.cvtColor(spoof_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(real_img_rgb)
plt.title("Sample Real Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(spoof_img_rgb)
plt.title("Sample Spoof Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Check image dimensions and channels
# This step checks the original size and number of channels
# of sample Real and Spoof images before preprocessing.

print("Real Image Shape:", real_img.shape)
print("Spoof Image Shape:", spoof_img.shape)

print("\nReal Image:")
print("Height:", real_img.shape[0])
print("Width :", real_img.shape[1])
print("Channels:", real_img.shape[2])

print("\nSpoof Image:")
print("Height:", spoof_img.shape[0])
print("Width :", spoof_img.shape[1])
print("Channels:", spoof_img.shape[2])

In [ ]:
# Step 6: Check corrupted or unreadable images
# This step verifies whether all dataset images can be loaded correctly.

import os
import cv2

corrupted_images = []
total_checked = 0

for split_folder in [
    "LCC_FASD_training",
    "LCC_FASD_development",
    "LCC_FASD_evaluation"
]:
    for label in ["real", "spoof"]:

        folder_path = os.path.join(BASE_PATH, split_folder, label)

        for filename in os.listdir(folder_path):

            if filename.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):

                image_path = os.path.join(folder_path, filename)

                img = cv2.imread(image_path)

                total_checked += 1

                if img is None:
                    corrupted_images.append(image_path)


print("Total Images Checked:", total_checked)
print("Corrupted / Unreadable Images:", len(corrupted_images))

if len(corrupted_images) == 0:
    print("Result: No corrupted images found.")
else:
    print("\nCorrupted Images:")
    for path in corrupted_images[:10]:
        print(path)

In [ ]:
# Step 7A: Download OpenCV DNN face detector model
# No additional Python package installation is required.

import os
import urllib.request

MODEL_DIR = "/kaggle/working/face_detector"
os.makedirs(MODEL_DIR, exist_ok=True)

PROTO_PATH = os.path.join(MODEL_DIR, "deploy.prototxt")
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "res10_300x300_ssd_iter_140000.caffemodel"
)

PROTO_URL = (
    "https://raw.githubusercontent.com/opencv/opencv/"
    "master/samples/dnn/face_detector/deploy.prototxt"
)

MODEL_URL = (
    "https://raw.githubusercontent.com/opencv/opencv_3rdparty/"
    "dnn_samples_face_detector_20170830/"
    "res10_300x300_ssd_iter_140000.caffemodel"
)

if not os.path.exists(PROTO_PATH):
    urllib.request.urlretrieve(PROTO_URL, PROTO_PATH)

if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

print("Face detector files downloaded successfully.")
print("Prototxt:", os.path.exists(PROTO_PATH))
print("Model:", os.path.exists(MODEL_PATH))

In [ ]:
# Step 7B: Detect and crop face using OpenCV DNN
# The detector works on the original color image before resizing
# and grayscale conversion.

import cv2
import numpy as np
import matplotlib.pyplot as plt

face_net = cv2.dnn.readNetFromCaffe(
    PROTO_PATH,
    MODEL_PATH
)

def detect_and_crop_face(image_path, confidence_threshold=0.5):

    img = cv2.imread(image_path)

    if img is None:
        return None

    h, w = img.shape[:2]

    blob = cv2.dnn.blobFromImage(
        cv2.resize(img, (300, 300)),
        1.0,
        (300, 300),
        (104.0, 177.0, 123.0)
    )

    face_net.setInput(blob)
    detections = face_net.forward()

    best_confidence = 0
    best_box = None

    for i in range(detections.shape[2]):

        confidence = detections[0, 0, i, 2]

        if confidence > best_confidence:

            box = detections[0, 0, i, 3:7] * np.array(
                [w, h, w, h]
            )

            best_box = box.astype(int)
            best_confidence = confidence

    if best_box is None or best_confidence < confidence_threshold:
        return None

    x1, y1, x2, y2 = best_box

    # Keep coordinates inside image boundary
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    # Add a small margin around the face
    face_w = x2 - x1
    face_h = y2 - y1

    margin_x = int(face_w * 0.15)
    margin_y = int(face_h * 0.15)

    x1 = max(0, x1 - margin_x)
    y1 = max(0, y1 - margin_y)
    x2 = min(w, x2 + margin_x)
    y2 = min(h, y2 + margin_y)

    face = img[y1:y2, x1:x2]

    return face


real_face = detect_and_crop_face(real_image_path)
spoof_face = detect_and_crop_face(spoof_image_path)

plt.figure(figsize=(10, 4))

if real_face is not None:
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(real_face, cv2.COLOR_BGR2RGB))
    plt.title("Cropped Real Face")
    plt.axis("off")
else:
    print("Real face was not detected.")

if spoof_face is not None:
    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(spoof_face, cv2.COLOR_BGR2RGB))
    plt.title("Cropped Spoof Face")
    plt.axis("off")
else:
    print("Spoof face was not detected.")

plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Final Image Preprocessing
# Cropped Face -> Resize -> Grayscale -> Normalize

import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_SIZE = 256

def preprocess_face(face):

    if face is None:
        return None

    # 1. Resize all cropped faces to the same size
    resized = cv2.resize(
        face,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )

    # 2. Convert image to grayscale
    gray = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2GRAY
    )

    # 3. Normalize pixel values from 0-255 to 0-1
    normalized = gray.astype(np.float32) / 255.0

    return normalized


# Apply preprocessing
real_processed = preprocess_face(real_face)
spoof_processed = preprocess_face(spoof_face)


# Check preprocessing results
print("Real processed shape :", real_processed.shape)
print("Spoof processed shape:", spoof_processed.shape)

print("\nReal pixel range :",
      real_processed.min(), "to", real_processed.max())

print("Spoof pixel range:",
      spoof_processed.min(), "to", spoof_processed.max())


# Display final preprocessed images
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(real_processed, cmap="gray")
plt.title("Preprocessed Real Face")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(spoof_processed, cmap="gray")
plt.title("Preprocessed Spoof Face")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Validate preprocessing on multiple random samples
# Random Real and Spoof images are selected to check whether
# face detection, cropping and preprocessing work consistently.

import random
import matplotlib.pyplot as plt

NUM_SAMPLES = 4

real_files = [
    os.path.join(real_folder, f)
    for f in os.listdir(real_folder)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

spoof_files = [
    os.path.join(spoof_folder, f)
    for f in os.listdir(spoof_folder)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

random.seed(42)

selected_real = random.sample(real_files, NUM_SAMPLES)
selected_spoof = random.sample(spoof_files, NUM_SAMPLES)

plt.figure(figsize=(12, 8))

# Real samples
for i, path in enumerate(selected_real):

    face = detect_and_crop_face(path)

    if face is not None:
        processed = preprocess_face(face)

        plt.subplot(2, NUM_SAMPLES, i + 1)
        plt.imshow(processed, cmap="gray")
        plt.title("Real")
        plt.axis("off")


# Spoof samples
for i, path in enumerate(selected_spoof):

    face = detect_and_crop_face(path)

    if face is not None:
        processed = preprocess_face(face)

        plt.subplot(2, NUM_SAMPLES, NUM_SAMPLES + i + 1)
        plt.imshow(processed, cmap="gray")
        plt.title("Spoof")
        plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# Step 10: Validate preprocessing pipeline on the full dataset
# Each image goes through:
# Face Detection -> Face Crop -> Resize -> Grayscale -> Normalization
# The final report shows successful and failed preprocessing samples.

import os
import pandas as pd

results = []

for split_name, split_folder in splits.items():

    for label in ["real", "spoof"]:

        folder_path = os.path.join(
            BASE_PATH,
            split_folder,
            label
        )

        success = 0
        failed = 0
        total = 0

        image_files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
        ]

        print(f"Processing {split_name} - {label} ...")

        for filename in image_files:

            total += 1

            image_path = os.path.join(
                folder_path,
                filename
            )

            # Face detection and crop
            face = detect_and_crop_face(image_path)

            if face is None:
                failed += 1
                continue

            # Resize + Grayscale + Normalize
            processed = preprocess_face(face)

            if (
                processed is not None
                and processed.shape == (256, 256)
                and processed.min() >= 0
                and processed.max() <= 1
            ):
                success += 1
            else:
                failed += 1

        results.append({
            "Split": split_name,
            "Class": label.capitalize(),
            "Total": total,
            "Successfully Preprocessed": success,
            "Failed Face Detection": failed
        })


preprocessing_summary = pd.DataFrame(results)

display(preprocessing_summary)

print("\nTotal Images:",
      preprocessing_summary["Total"].sum())

print("Successfully Preprocessed:",
      preprocessing_summary["Successfully Preprocessed"].sum())

print("Failed:",
      preprocessing_summary["Failed Face Detection"].sum())

# Dataset 2: CASIA-FASD

CASIA-FASD is used as the second benchmark dataset for this face anti-spoofing research.

The purpose of using a second dataset is to evaluate the preprocessing pipeline on different face presentation attack conditions and to support broader validation of the proposed frequency-domain approach.

For this dataset, the preprocessing workflow will include:

- Dataset structure inspection
- Real and spoof sample analysis
- Image/video format inspection
- Data integrity checking
- Face detection and cropping
- Resize to a fixed resolution
- Grayscale conversion
- Pixel normalization

After preprocessing, the prepared facial samples will be used for frequency-domain analysis using 2D FFT.

In [ ]:
# Dataset 2 - Step 1: Inspect CASIA-FASD directory structure
# This step identifies the exact Kaggle path, folders and file organization
# before starting preprocessing.

import os

print("Available Kaggle inputs:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 5:
        indent = "    " * level
        print(indent + os.path.basename(root) + "/")

In [ ]:
# Dataset 2 - Step 2: Find CASIA-FASD path and inspect folder structure
# This step identifies the exact dataset location and
# shows how the CASIA-FASD dataset is organized.

import os

CASIA_ROOT = None

# Find exact CASIA-FASD root path
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root).lower() == "casia-fasd":
        CASIA_ROOT = root
        break

print("CASIA-FASD Root Path:")
print(CASIA_ROOT)

print("\nCASIA-FASD Directory Structure:\n")

# Display folder structure
for root, dirs, files in os.walk(CASIA_ROOT):

    level = root.replace(CASIA_ROOT, "").count(os.sep)

    if level <= 3:

        indent = "    " * level

        print(
            indent +
            os.path.basename(root) +
            f"/  ({len(files)} files)"
        )

In [ ]:
# Dataset 2 - Step 3: Check total files and file types
# This step identifies whether the dataset contains
# images, videos, extracted frames, or other files.

from collections import Counter

file_types = Counter()
total_files = 0

for root, dirs, files in os.walk(CASIA_ROOT):

    for filename in files:

        total_files += 1

        ext = os.path.splitext(filename)[1].lower()

        if ext == "":
            ext = "[No Extension]"

        file_types[ext] += 1


print("Total CASIA-FASD Files:", total_files)

print("\nFile Type Distribution:")

for ext, count in file_types.most_common():
    print(f"{ext}: {count}")

In [ ]:
# Dataset 2 - Step 4: Inspect sample file paths
# This step helps understand Real/Spoof labels,
# Train/Test structure, and attack organization.

sample_files = []

for root, dirs, files in os.walk(CASIA_ROOT):

    for filename in files:

        sample_files.append(
            os.path.join(root, filename)
        )

        if len(sample_files) >= 30:
            break

    if len(sample_files) >= 30:
        break


print("Sample CASIA-FASD File Paths:\n")

for path in sample_files:
    print(path)

In [ ]:
# Dataset 2 - Step 5: Count Live and Spoof images
# This step checks the class distribution in Train and Test sets.

import os
import pandas as pd

CASIA_DATA = os.path.join(CASIA_ROOT, "casia-fasd")

if not os.path.isdir(CASIA_DATA):
    CASIA_DATA = CASIA_ROOT

casia_folders = {
    "Train Live": os.path.join(CASIA_DATA, "train", "live"),
    "Train Spoof": os.path.join(CASIA_DATA, "train", "spoof"),
    "Test Live": os.path.join(CASIA_DATA, "test", "live"),
    "Test Spoof": os.path.join(CASIA_DATA, "test", "spoof")
}

data = []

for name, folder in casia_folders.items():

    count = len([
        f for f in os.listdir(folder)
        if f.lower().endswith(".png")
    ])

    split, label = name.split()

    data.append({
        "Split": split,
        "Class": label,
        "Images": count
    })

casia_counts = pd.DataFrame(data)

display(casia_counts)

print("Total Images:", casia_counts["Images"].sum())

In [ ]:
# Dataset 2 - Step 6: Display sample Live and Spoof images
# This visually verifies both classes before preprocessing.

import cv2
import matplotlib.pyplot as plt

train_live_folder = casia_folders["Train Live"]
train_spoof_folder = casia_folders["Train Spoof"]

live_image_path = os.path.join(
    train_live_folder,
    os.listdir(train_live_folder)[0]
)

spoof_image_path_casia = os.path.join(
    train_spoof_folder,
    os.listdir(train_spoof_folder)[0]
)

live_img = cv2.imread(live_image_path)
casia_spoof_img = cv2.imread(spoof_image_path_casia)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(live_img, cv2.COLOR_BGR2RGB))
plt.title("CASIA Live Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(casia_spoof_img, cv2.COLOR_BGR2RGB))
plt.title("CASIA Spoof Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Dataset 2 - Step 7: Check original image dimensions and channels
# This is done before face cropping and resizing.

print("Live Image Shape :", live_img.shape)
print("Spoof Image Shape:", casia_spoof_img.shape)

print("\nLive Image")
print("Height  :", live_img.shape[0])
print("Width   :", live_img.shape[1])
print("Channels:", live_img.shape[2])

print("\nSpoof Image")
print("Height  :", casia_spoof_img.shape[0])
print("Width   :", casia_spoof_img.shape[1])
print("Channels:", casia_spoof_img.shape[2])

In [ ]:
# Dataset 2 - Step 8: Check corrupted or unreadable images
# All PNG images are tested to confirm that they can be loaded correctly.

import cv2

corrupted_casia = []
total_checked_casia = 0

for folder_name, folder_path in casia_folders.items():

    print("Checking:", folder_name)

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".png"):

            image_path = os.path.join(folder_path, filename)

            img = cv2.imread(image_path)

            total_checked_casia += 1

            if img is None:
                corrupted_casia.append(image_path)


print("\nTotal Images Checked:", total_checked_casia)
print("Corrupted / Unreadable Images:", len(corrupted_casia))

if len(corrupted_casia) == 0:
    print("Result: No corrupted images found.")
else:
    print("Result: Some corrupted images were found.")

In [ ]:
# Dataset 2 - Step 9: Preprocess CASIA-FASD sample images
# Images are already face-focused and 112x112,
# so we only apply Grayscale -> Normalization.

import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_casia_image(image_path):

    # Read image
    img = cv2.imread(image_path)

    if img is None:
        return None

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Normalize pixel values from 0-255 to 0-1
    normalized = gray.astype(np.float32) / 255.0

    return normalized


# Preprocess one Live and one Spoof sample
casia_live_processed = preprocess_casia_image(live_image_path)
casia_spoof_processed = preprocess_casia_image(spoof_image_path_casia)


print("Live processed shape :", casia_live_processed.shape)
print("Spoof processed shape:", casia_spoof_processed.shape)

print("\nLive pixel range :",
      casia_live_processed.min(), "to",
      casia_live_processed.max())

print("Spoof pixel range:",
      casia_spoof_processed.min(), "to",
      casia_spoof_processed.max())


# Display preprocessed samples
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(casia_live_processed, cmap="gray")
plt.title("Preprocessed CASIA Live")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(casia_spoof_processed, cmap="gray")
plt.title("Preprocessed CASIA Spoof")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Dataset 2 - Step 10: Validate preprocessing on multiple samples
# Random Live and Spoof images are checked to confirm
# that preprocessing works consistently.

import random
import os
import matplotlib.pyplot as plt

NUM_SAMPLES = 4

live_files = [
    os.path.join(train_live_folder, f)
    for f in os.listdir(train_live_folder)
    if f.lower().endswith(".png")
]

spoof_files_casia = [
    os.path.join(train_spoof_folder, f)
    for f in os.listdir(train_spoof_folder)
    if f.lower().endswith(".png")
]

random.seed(42)

selected_live = random.sample(live_files, NUM_SAMPLES)
selected_spoof = random.sample(spoof_files_casia, NUM_SAMPLES)

plt.figure(figsize=(12, 8))

# Live samples
for i, path in enumerate(selected_live):

    processed = preprocess_casia_image(path)

    plt.subplot(2, NUM_SAMPLES, i + 1)
    plt.imshow(processed, cmap="gray")
    plt.title("Live")
    plt.axis("off")


# Spoof samples
for i, path in enumerate(selected_spoof):

    processed = preprocess_casia_image(path)

    plt.subplot(2, NUM_SAMPLES, NUM_SAMPLES + i + 1)
    plt.imshow(processed, cmap="gray")
    plt.title("Spoof")
    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# Dataset 2 - Step 11: Validate preprocessing on the full CASIA-FASD dataset
# Each image is converted to grayscale and normalized.
# Final shape and pixel range are checked.

import pandas as pd

casia_results = []

for folder_name, folder_path in casia_folders.items():

    successful = 0
    failed = 0

    image_files = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".png")
    ]

    print("Processing:", folder_name)

    for filename in image_files:

        image_path = os.path.join(folder_path, filename)

        processed = preprocess_casia_image(image_path)

        if (
            processed is not None
            and processed.shape == (112, 112)
            and processed.min() >= 0
            and processed.max() <= 1
        ):
            successful += 1
        else:
            failed += 1

    split, label = folder_name.split()

    casia_results.append({
        "Split": split,
        "Class": label,
        "Total": len(image_files),
        "Successfully Preprocessed": successful,
        "Failed": failed
    })


casia_preprocessing_summary = pd.DataFrame(casia_results)

display(casia_preprocessing_summary)

print("\nTotal Images:",
      casia_preprocessing_summary["Total"].sum())

print("Successfully Preprocessed:",
      casia_preprocessing_summary["Successfully Preprocessed"].sum())

print("Failed:",
      casia_preprocessing_summary["Failed"].sum())

In [ ]:
# Cell 1 — Imports and paths; works in a fresh Kaggle session
import os, re, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 112
MAX_PER_CLASS = 100  # use None AFTER pilot validation to process all images
OUTPUT_DIR = Path('/kaggle/working/fft_baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LCC_ROOT = Path('/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD')
CASIA_ROOT = Path('/kaggle/input/datasets/immada/casia-fasd/casia-fasd')
if not CASIA_ROOT.exists():
    CASIA_ROOT = Path('/kaggle/input/datasets/immada/casia-fasd')
for root in (LCC_ROOT, CASIA_ROOT):
    assert root.exists(), f'Dataset not found: {root}'

LCC_SPLITS = {
    'training': 'LCC_FASD_training',
    'development': 'LCC_FASD_development',
    'evaluation': 'LCC_FASD_evaluation'
}
CASIA_SPLITS = {'train': 'train', 'test': 'test'}
print('LCC:', LCC_ROOT, '\nCASIA:', CASIA_ROOT)
print('OpenCV version:', cv2.__version__)


In [ ]:
# Cell 2 — Explicit manifests and CASIA video-group audit
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

def get_paths(directory):
    assert directory.is_dir(), f'Missing expected class folder: {directory}'
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT)

def casia_video_id(path):
    # Observed names include s9v2f27.png; treat (s, v) as a provisional group,
    # and verify against the dataset's original video-level protocol.
    m = re.match(r'^s(\d+)v(\d+)f(\d+)$', path.stem, re.IGNORECASE)
    return f's{m.group(1)}v{m.group(2)}' if m else None

manifest_rows = []
for split, folder in LCC_SPLITS.items():
    for cls in ('real', 'spoof'):
        for p in get_paths(LCC_ROOT / folder / cls):
            manifest_rows.append(dict(dataset='LCC', split=split, label=cls,
                                      path=str(p), video_id=None))
for split, folder in CASIA_SPLITS.items():
    for cls in ('live', 'spoof'):
        for p in get_paths(CASIA_ROOT / folder / cls):
            manifest_rows.append(dict(dataset='CASIA', split=split,
                                      label='real' if cls == 'live' else 'spoof',
                                      path=str(p), video_id=casia_video_id(p)))
manifest = pd.DataFrame(manifest_rows)
print(manifest.groupby(['dataset', 'split', 'label']).size().rename('files').to_string())
casia = manifest[manifest.dataset == 'CASIA']
print('CASIA files with unparsed video ID:', casia.video_id.isna().sum())
# Video IDs alone may not be globally unique across attack types/official partitions.
# Do NOT merge or re-split official CASIA train/test from these IDs without original protocol metadata.
manifest.to_csv(OUTPUT_DIR / 'image_manifest.csv', index=False)
print('Saved:', OUTPUT_DIR / 'image_manifest.csv')


In [ ]:
# Cell 3 — Download and load the same OpenCV detector as your earlier LCC code
MODEL_DIR = Path('/kaggle/working/face_detector')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROTO = MODEL_DIR / 'deploy.prototxt'
WEIGHTS = MODEL_DIR / 'res10_300x300_ssd_iter_140000.caffemodel'
FILES = [
    (PROTO, 'https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt'),
    (WEIGHTS, 'https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel')
]
for destination, url in FILES:
    if not destination.exists() or destination.stat().st_size == 0:
        urllib.request.urlretrieve(url, destination)
face_net = cv2.dnn.readNetFromCaffe(str(PROTO), str(WEIGHTS))
print('Face detector loaded.')


In [ ]:
# Cell 4 — One preprocessing interface for both datasets
# LCC uses face cropping. CASIA packaged frames are already 112x112 face-focused;
# do NOT crop CASIA with the LCC detector, which would alter dataset treatment.
def crop_lcc_face(bgr, confidence_threshold=0.5):
    h, w = bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(bgr, (300, 300)),
                                 1.0, (300, 300), (104.0, 177.0, 123.0))
    face_net.setInput(blob)
    detections = face_net.forward()
    if detections.shape[2] == 0:
        return None
    idx = int(np.argmax(detections[0, 0, :, 2]))
    conf = float(detections[0, 0, idx, 2])
    if conf < confidence_threshold:
        return None
    x1, y1, x2, y2 = (detections[0, 0, idx, 3:7] * [w, h, w, h]).astype(int)
    x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
    if x2 <= x1 or y2 <= y1:
        return None
    mx, my = int((x2-x1)*0.15), int((y2-y1)*0.15)
    x1,y1,x2,y2 = max(0,x1-mx),max(0,y1-my),min(w,x2+mx),min(h,y2+my)
    return bgr[y1:y2, x1:x2]

def read_preprocessed(path, dataset):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 'unreadable'
    if dataset == 'LCC':
        bgr = crop_lcc_face(bgr)
        if bgr is None or bgr.size == 0:
            return None, 'face_not_detected'
    # Resize once from the cropped original; CASIA is 112x112 already.
    if bgr.shape[:2] != (IMG_SIZE, IMG_SIZE):
        interpolation = cv2.INTER_AREA if min(bgr.shape[:2]) >= IMG_SIZE else cv2.INTER_LINEAR
        bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=interpolation)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    x = gray.astype(np.float32) / 255.0
    if x.shape != (IMG_SIZE, IMG_SIZE) or not np.isfinite(x).all() or x.min()<0 or x.max()>1:
        return None, 'preprocess_validation_failed'
    return x, 'ok'


In [ ]:
# Cell 5 — Reproducible, windowed FFT with a numerically stable log magnitude
# A Hann window reduces artificial high-frequency signals from square crop borders.
HANN2D = np.outer(np.hanning(IMG_SIZE), np.hanning(IMG_SIZE)).astype(np.float32)

def fft_magnitude(gray, apply_window=True):
    assert gray.shape == (IMG_SIZE, IMG_SIZE)
    x = gray * HANN2D if apply_window else gray
    fft = np.fft.fftshift(np.fft.fft2(x))
    return np.log1p(np.abs(fft)).astype(np.float32), fft

def show_fft_pair(df, dataset, split):
    chosen = {}
    for label in ('real', 'spoof'):
        subset = df[(df.dataset==dataset) & (df.split==split) & (df.label==label)]
        for row in subset.sample(frac=1, random_state=SEED).itertuples(index=False):
            gray, reason = read_preprocessed(row.path, dataset)
            if gray is not None:
                chosen[label] = (gray, fft_magnitude(gray)[0], row.path)
                break
        assert label in chosen, f'No preprocessable {dataset}/{split}/{label} samples found'
    # Use a shared FFT intensity scale so visual comparisons aren't misleading.
    a, b = chosen['real'][1], chosen['spoof'][1]
    low, high = min(a.min(), b.min()), max(a.max(), b.max())
    fig, axes = plt.subplots(2, 2, figsize=(9, 8))
    for i, label in enumerate(('real','spoof')):
        gray, mag, path = chosen[label]
        axes[i,0].imshow(gray, cmap='gray', vmin=0, vmax=1)
        axes[i,0].set_title(f'{dataset} {label} (112×112)')
        axes[i,1].imshow(mag, cmap='gray', vmin=low, vmax=high)
        axes[i,1].set_title(f'{dataset} {label} — windowed FFT')
        print(label, Path(path).name, '| FFT min/max:',float(mag.min()),float(mag.max()))
    for ax in axes.flat: ax.axis('off')
    fig.tight_layout()
    out = OUTPUT_DIR / f'{dataset.lower()}_fft_examples.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved:', out)

show_fft_pair(manifest, 'LCC', 'training')
show_fft_pair(manifest, 'CASIA', 'train')


In [ ]:
# Cell 6 — Interpretable radial FFT power-band features
# Divide by total non-DC power so illumination/contrast differences don't
# dominate simple raw-energy features. Entropy describes power dispersion.
y, x = np.indices((IMG_SIZE, IMG_SIZE), dtype=np.float32)
center = (IMG_SIZE - 1) / 2.0
radius = np.sqrt((x-center)**2 + (y-center)**2)
radius /= radius.max()  # 0 at center, 1 in farthest corners
BANDS = {'low': (0.0,0.15), 'mid': (0.15,0.40), 'high': (0.40,1.00001)}

# Even-sized fftshift has DC at (N/2,N/2), rather than ((N-1)/2,...).
# Rebuild exact radius relative to DC index to avoid radial mask asymmetry.
cy = cx = IMG_SIZE // 2
radius = np.sqrt((x-cx)**2 + (y-cy)**2)
radius /= radius.max()
NON_DC = np.ones((IMG_SIZE, IMG_SIZE), dtype=bool)
NON_DC[cy,cx] = False

def extract_fft_features(gray):
    mag, fft = fft_magnitude(gray)
    power = np.abs(fft)**2
    power = power.astype(np.float64)
    total = float(power[NON_DC].sum())
    if not np.isfinite(total) or total <= 1e-12:
        return None  # constant/degenerate input
    features = {}
    for name,(lo,hi) in BANDS.items():
        mask = (radius >= lo) & (radius < hi) & NON_DC
        features[f'{name}_power_ratio'] = float(power[mask].sum()/total)
        features[f'{name}_mean_log_magnitude'] = float(mag[mask].mean())
        features[f'{name}_std_log_magnitude'] = float(mag[mask].std())
    # Full non-DC normalized spectral entropy (0..1).
    p = power[NON_DC] / total
    p = p[p > 0]
    features['spectral_entropy'] = float(-np.sum(p*np.log(p))/np.log(NON_DC.sum()))
    features['fft_log_mean'] = float(mag[NON_DC].mean())
    features['fft_log_std'] = float(mag[NON_DC].std())
    assert np.isfinite(list(features.values())).all()
    assert abs(sum(features[f'{name}_power_ratio'] for name in BANDS)-1.0)<1e-6
    return features

# Self-test independent of the Kaggle dataset
rng = np.random.default_rng(SEED)
probe = rng.random((IMG_SIZE,IMG_SIZE), dtype=np.float32)
test_features = extract_fft_features(probe)
assert len(test_features) == 12
print('FFT extraction self-test passed. Feature count:', len(test_features))
print(pd.Series(test_features).round(4).to_string())


In [ ]:
# Cell 7 — Small, deterministic PILOT export, official splits kept separate
# No train/test mixing, balancing, standardization, classification or thresholds here.
# Later split-aware training must fit scalers only on training data.

def select_pilot(group):
    # Fixed shuffle to get a reproducible class-level sample.
    group = group.sample(frac=1, random_state=SEED)
    return group if MAX_PER_CLASS is None else group.head(MAX_PER_CLASS)

selected = (manifest.groupby(['dataset','split','label'], group_keys=False)
            .apply(select_pilot))
print('Requested pilot rows:', len(selected))
feature_rows, failures = [], []
for row in selected.itertuples(index=False):
    gray, status = read_preprocessed(row.path, row.dataset)
    if gray is None:
        failures.append(dict(dataset=row.dataset, split=row.split, label=row.label,
                             path=row.path, reason=status))
        continue
    features = extract_fft_features(gray)
    if features is None:
        failures.append(dict(dataset=row.dataset, split=row.split, label=row.label,
                             path=row.path, reason='degenerate_fft'))
        continue
    feature_rows.append(dict(dataset=row.dataset, split=row.split, label=row.label,
                             path=row.path, video_id=row.video_id, **features))

feature_df = pd.DataFrame(feature_rows)
failure_df = pd.DataFrame(failures,columns=['dataset','split','label','path','reason'])
assert len(feature_df)+len(failure_df)==len(selected)
if not feature_df.empty:
    for (dataset,split), chunk in feature_df.groupby(['dataset','split']):
        dest = OUTPUT_DIR / f'{dataset.lower()}_{split}_fft_features.csv'
        chunk.to_csv(dest, index=False)
        print('Saved:',dest,'rows:',len(chunk))
failure_df.to_csv(OUTPUT_DIR / 'fft_preprocessing_failures.csv', index=False)
print('Failures:',len(failure_df))
print(feature_df.groupby(['dataset','split','label']).size().rename('exported').to_string())
print('Outputs:', OUTPUT_DIR)


In [ ]:
# Cell A: Experiment configuration

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import time
import json
import joblib
import warnings

# False = pilot experiment
# True = full dataset experiment
FULL_RUN = False

SEED = 42

# Limit the number of CASIA training frames per video
TRAIN_FRAMES_PER_VIDEO = 10

# Restrict SVM training size for computational efficiency
SVM_TRAIN_CAP = 5000

# DCT configuration
DCT_SIZE = 8

# Output directory
PIPELINE_DIR = OUTPUT_DIR / "full_research"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)


def select_phase_rows(group):
    if FULL_RUN:
        return group

    return group.sample(
        n=min(30, len(group)),
        random_state=SEED
    )


phase_manifest = pd.concat(
    [
        select_phase_rows(group)
        for _, group in manifest.groupby(
            ["dataset", "split", "label"]
        )
    ],
    ignore_index=True
)

assert len(phase_manifest) > 0

print(
    "RUN MODE:",
    "FULL DATASET"
    if FULL_RUN
    else "PILOT EXPERIMENT"
)

print(
    phase_manifest.groupby(
        ["dataset", "split", "label"]
    ).size()
)

In [ ]:
# Cell B: DCT and combined FFT+DCT features

def extract_dct_features(gray):

    dct = cv2.dct(
        np.asarray(gray, dtype=np.float32)
    )

    block = np.abs(
        dct[:DCT_SIZE, :DCT_SIZE]
    ).astype(np.float64)

    block[0, 0] = 0.0

    rr, cc = np.indices(block.shape)
    order = rr + cc

    total = float(np.sum(block ** 2))

    if total <= 1e-14 or not np.isfinite(total):
        return None

    out = {}

    bands = [
        ("dct_low", (order > 0) & (order <= 3)),
        ("dct_mid", (order > 3) & (order <= 7)),
        ("dct_high", order > 7)
    ]

    for name, mask in bands:

        vals = block[mask]

        out[name + "_energy_ratio"] = float(
            np.sum(vals ** 2) / total
        )

        out[name + "_mean_abs"] = float(
            np.mean(vals)
        )

    energy_sum = sum(
        out[k + "_energy_ratio"]
        for k in ("dct_low", "dct_mid", "dct_high")
    )

    assert abs(energy_sum - 1.0) < 1e-6

    return out


def all_features(gray):

    fft_features = extract_fft_features(gray)
    dct_features = extract_dct_features(gray)

    if fft_features is None or dct_features is None:
        return None

    return {
        **fft_features,
        **dct_features
    }


# Numeric self-test

sample = np.random.default_rng(
    SEED
).random(
    (IMG_SIZE, IMG_SIZE)
).astype(np.float32)

probe = all_features(sample)

assert probe is not None
assert len(probe) == 18
assert np.isfinite(list(probe.values())).all()

print("FFT + DCT feature extraction test passed")
print("Total features:", len(probe))
print(sorted(probe.keys()))

In [ ]:
# Cell C: Restartable feature extraction

import hashlib

def fingerprint(paths):
    return hashlib.sha256(
        "\n".join(map(str, paths)).encode()
    ).hexdigest()


feature_parts = []
failure_parts = []

for (dataset, split, label), chunk in phase_manifest.groupby(
    ["dataset", "split", "label"],
    sort=True
):

    chunk = chunk.sort_values("path")

    stem = (
        f"{dataset.lower()}_{split}_{label}_"
        + ("full" if FULL_RUN else "pilot")
    )

    dest = PIPELINE_DIR / f"{stem}_features.csv"
    bad_dest = PIPELINE_DIR / f"{stem}_failures.csv"
    marker = PIPELINE_DIR / f"{stem}_complete.json"

    sign = fingerprint(chunk.path.tolist())

    # Restore completed checkpoint

    if (
        marker.exists()
        and dest.exists()
        and bad_dest.exists()
    ):

        info = json.loads(marker.read_text())

        if (
            info.get("fingerprint") == sign
            and info.get("expected") == len(chunk)
            and info.get("img_size") == IMG_SIZE
        ):

            good = pd.read_csv(dest)
            bad = pd.read_csv(bad_dest)

            if len(good) + len(bad) == len(chunk):

                print("RESTORED:", stem)

                feature_parts.append(good)
                failure_parts.append(bad)

                continue

    rows = []
    bad_rows = []

    start = time.perf_counter()

    for index, row in enumerate(
        chunk.itertuples(index=False),
        start=1
    ):

        gray, status = read_preprocessed(
            row.path,
            row.dataset
        )

        extracted = (
            all_features(gray)
            if gray is not None
            else None
        )

        if extracted is None:

            bad_rows.append({
                "dataset": dataset,
                "split": split,
                "label": label,
                "path": row.path,
                "reason": (
                    status if gray is None
                    else "invalid_fft_or_dct"
                )
            })

        else:

            rows.append({
                "dataset": dataset,
                "split": split,
                "label": label,
                "path": row.path,
                "video_id": row.video_id,
                **extracted
            })

        if index % 2000 == 0:

            print(
                stem,
                index,
                "/",
                len(chunk)
            )

    good = pd.DataFrame(rows)

    bad = pd.DataFrame(
        bad_rows,
        columns=[
            "dataset",
            "split",
            "label",
            "path",
            "reason"
        ]
    )

    assert len(good) + len(bad) == len(chunk)

    if len(good):

        good.to_csv(
            dest,
            index=False
        )

    else:

        pd.DataFrame(
            columns=[
                "dataset",
                "split",
                "label",
                "path",
                "video_id"
            ] + list(probe)
        ).to_csv(
            dest,
            index=False
        )

    bad.to_csv(
        bad_dest,
        index=False
    )

    marker.write_text(
        json.dumps({
            "expected": len(chunk),
            "fingerprint": sign,
            "passed": len(good),
            "failed": len(bad),
            "img_size": IMG_SIZE
        })
    )

    print(
        "SAVED:",
        stem,
        len(good),
        "valid,",
        len(bad),
        "failed"
    )

    feature_parts.append(
        pd.read_csv(dest)
    )

    failure_parts.append(
        pd.read_csv(bad_dest)
    )


features = pd.concat(
    feature_parts,
    ignore_index=True
)

failures = pd.concat(
    failure_parts,
    ignore_index=True
)

mode = "full" if FULL_RUN else "pilot"

features.to_csv(
    PIPELINE_DIR / f"all_features_{mode}.csv",
    index=False
)

failures.to_csv(
    PIPELINE_DIR / f"failures_{mode}.csv",
    index=False
)

print("EXTRACTION SUMMARY")

print(
    features.groupby(
        ["dataset", "split", "label"]
    ).size()
)

print("Failures:", len(failures))

In [ ]:

# ============================================================
# CELL D: CASIA Filename, Subject & Feature Validation
# ============================================================

import re
import json
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Extract provisional CASIA subject IDs
# ------------------------------------------------------------
# Supported filename examples:
# s9v2f27.png
# s10vHR_2f157.png
# bs11v1f38.png
# fs10v1f14.png

def get_subject(path):

    filename = Path(path).stem

    pattern = r"^(?:b|f)?s(\d+)v(?:HR_)?\d+f\d+$"

    match = re.fullmatch(
        pattern,
        filename,
        flags=re.IGNORECASE
    )

    if match:
        return "s" + match.group(1)

    return None


# ------------------------------------------------------------
# 2. Select CASIA samples
# ------------------------------------------------------------

cas = features[
    features["dataset"] == "CASIA"
].copy()

cas["subject_guess"] = cas["path"].map(
    get_subject
)

print("Total CASIA samples:", len(cas))


# ------------------------------------------------------------
# 3. Validate filename formats
# ------------------------------------------------------------

unknown = cas[
    cas["subject_guess"].isna()
]

print("Unrecognized filenames:", len(unknown))

if len(unknown) > 0:

    print("\nUnrecognized filenames:")

    print(
        unknown["path"]
        .apply(lambda p: Path(p).name)
        .head(30)
        .to_string(index=False)
    )

    raise ValueError(
        "Unrecognized filenames remain. "
        "Review them before proceeding."
    )

print("Filename validation passed.")


# ------------------------------------------------------------
# 4. Check provisional subject overlap
# ------------------------------------------------------------

train_subjects = set(
    cas.loc[
        cas["split"] == "train",
        "subject_guess"
    ]
)

test_subjects = set(
    cas.loc[
        cas["split"] == "test",
        "subject_guess"
    ]
)

overlap = sorted(
    train_subjects.intersection(test_subjects)
)

print("\nTrain subjects:", len(train_subjects))
print("Test subjects:", len(test_subjects))
print("Subject overlap:", overlap)


# ------------------------------------------------------------
# 5. Validate extracted features
# ------------------------------------------------------------

feature_names = list(probe.keys())

assert len(features) > 0, (
    "No extracted features available."
)

assert not features["path"].duplicated().any(), (
    "Duplicate file paths detected."
)

missing_columns = [
    name for name in feature_names
    if name not in features.columns
]

assert not missing_columns, (
    f"Missing feature columns: {missing_columns}"
)

feature_array = features[
    feature_names
].to_numpy(dtype=np.float64)

assert not np.isnan(feature_array).any(), (
    "Missing or NaN feature values detected."
)

assert np.isfinite(feature_array).all(), (
    "Infinite or non-finite feature values detected."
)

print("\nFeature validation passed.")
print("Total feature rows:", len(features))
print("Features per sample:", len(feature_names))


# ------------------------------------------------------------
# 6. Save audit report
# ------------------------------------------------------------

report = {
    "CASIA_total_samples": int(len(cas)),
    "unrecognized_filenames": int(len(unknown)),
    "train_subjects": len(train_subjects),
    "test_subjects": len(test_subjects),
    "subject_overlap": overlap,
    "LCC_subject_overlap": "NOT_VERIFIED",
    "subject_id_status": (
        "Provisional filename-derived IDs. "
        "Official dataset protocol verification required."
    ),
    "pilot": not FULL_RUN,
    "feature_rows": int(len(features)),
    "features_per_sample": len(feature_names),
    "feature_validation": "PASSED"
}

PIPELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    PIPELINE_DIR / "protocol_audit.json"
)

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("CELL D VALIDATION SUMMARY")
print("=" * 50)

print("CASIA samples:", len(cas))
print("Unrecognized filenames:", len(unknown))
print("Training subjects:", len(train_subjects))
print("Testing subjects:", len(test_subjects))
print("Potential subject overlap:", overlap)
print("Extracted feature rows:", len(features))
print("Features per sample:", len(feature_names))
print("Audit report:", report_path)

if overlap:
    print(
        "\nWARNING: Potential subject overlap detected. "
        "Review the dataset protocol before training."
    )
else:
    print(
        "\nNo subject overlap detected under the "
        "provisional filename convention."
    )

print("\nCell D execution completed.")


In [ ]:
from pathlib import Path
import pandas as pd

cas = features[
    features["dataset"] == "CASIA"
].copy()

cas["filename"] = cas["path"].apply(
    lambda p: Path(p).name
)

cas["subject_guess"] = cas["path"].map(
    get_subject
)

unknown = cas[
    cas["subject_guess"].isna()
]

print("Total CASIA samples:", len(cas))
print("Unrecognized filenames:", len(unknown))

print("\nFirst 20 unrecognized filenames:")
print(
    unknown["filename"].head(20).to_string(
        index=False
    )
)

print("\nFirst 10 full paths:")
print(
    unknown["path"].head(10).to_string(
        index=False
    )
)

In [ ]:

# ============================================================
# CELL E: Threshold Selection and Evaluation Metrics
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Select threshold using calibration data only
# ------------------------------------------------------------

def choose_threshold(scores, y):

    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=int)

    assert len(scores) == len(y)
    assert len(scores) > 0
    assert set(np.unique(y)) == {0, 1}
    assert np.isfinite(scores).all()

    # Generate candidate thresholds
    vals = np.unique(scores)

    if len(vals) == 1:
        thresholds = np.array([
            np.nextafter(vals[0], -np.inf),
            vals[0],
            np.nextafter(vals[0], np.inf)
        ])

    else:
        midpoints = (
            vals[:-1] + vals[1:]
        ) / 2.0

        thresholds = np.concatenate([
            [np.nextafter(vals[0], -np.inf)],
            midpoints,
            [np.nextafter(vals[-1], np.inf)]
        ])

    best_threshold = None
    best_key = None

    # Find threshold with minimum ACER
    for threshold in thresholds:

        predictions = (
            scores >= threshold
        ).astype(int)

        apcer = np.mean(
            predictions[y == 1] == 0
        )

        bpcer = np.mean(
            predictions[y == 0] == 1
        )

        acer = (apcer + bpcer) / 2.0

        key = (
            acer,
            abs(apcer - bpcer),
            float(threshold)
        )

        if best_key is None or key < best_key:

            best_key = key
            best_threshold = float(threshold)

    return best_threshold


# ------------------------------------------------------------
# 2. Calculate classification metrics
# ------------------------------------------------------------

def metrics(y, score, threshold):

    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)

    assert len(y) == len(score)
    assert len(y) > 0
    assert set(np.unique(y)) == {0, 1}
    assert np.isfinite(score).all()
    assert np.isfinite(threshold)

    predictions = (
        score >= threshold
    ).astype(int)

    # Confusion matrix:
    # TN = Real correctly predicted as Real
    # FP = Real incorrectly predicted as Spoof
    # FN = Spoof incorrectly predicted as Real
    # TP = Spoof correctly predicted as Spoof

    tn, fp, fn, tp = confusion_matrix(
        y,
        predictions,
        labels=[0, 1]
    ).ravel()

    # Attack Presentation Classification Error Rate
    apcer = fn / (tp + fn)

    # Bona Fide Presentation Classification Error Rate
    bpcer = fp / (tn + fp)

    # Average Classification Error Rate
    acer = (apcer + bpcer) / 2.0

    accuracy = accuracy_score(
        y,
        predictions
    )

    roc_auc = roc_auc_score(
        y,
        score
    )

    pr_auc = average_precision_score(
        y,
        score
    )

    return {

        "accuracy": float(accuracy),

        "APCER": float(apcer),

        "BPCER": float(bpcer),

        "ACER": float(acer),

        "ROC_AUC": float(roc_auc),

        "PR_AUC": float(pr_auc),

        "threshold": float(threshold),

        "real_n": int(tn + fp),

        "spoof_n": int(tp + fn),

        "TN": int(tn),

        "FP": int(fp),

        "FN": int(fn),

        "TP": int(tp)
    }


# ------------------------------------------------------------
# 3. Test evaluation functions
# ------------------------------------------------------------

# Synthetic test data (NOT research results)

test_y = np.array([
    0, 0, 0, 1, 1, 1
])

test_scores = np.array([
    0.10,
    0.20,
    0.30,
    0.70,
    0.80,
    0.90
])

# Select threshold

selected_threshold = choose_threshold(
    test_scores,
    test_y
)

# Evaluate

test_results = metrics(
    test_y,
    test_scores,
    selected_threshold
)


# ------------------------------------------------------------
# 4. Verify correct implementation
# ------------------------------------------------------------

assert test_results["accuracy"] == 1.0

assert test_results["APCER"] == 0.0

assert test_results["BPCER"] == 0.0

assert test_results["ACER"] == 0.0

assert test_results["ROC_AUC"] == 1.0


# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("=" * 50)
print("CELL E: EVALUATION METRICS TEST")
print("=" * 50)

print(
    "Selected threshold:",
    selected_threshold
)

for name, value in test_results.items():

    print(
        f"{name}: {value}"
    )

print("\nAll evaluation function tests passed.")

print(
    "IMPORTANT: These are synthetic test results, "
    "not actual face anti-spoofing performance."
)


In [ ]:

# ============================================================
# FIX: Recover CASIA Video IDs from Frame Filenames
# ============================================================

import re
import pandas as pd
from pathlib import Path

# Example:
# s9v2f27.png       -> s9v2
# s10vHR_2f157.png  -> s10vHR_2
# bs11v1f38.png     -> bs11v1
# fs10v1f14.png     -> fs10v1

def extract_casia_video_id(path, split):

    filename = Path(path).stem

    match = re.fullmatch(
        r"((?:b|f)?s\d+v(?:HR_)?\d+)f\d+",
        filename,
        flags=re.IGNORECASE
    )

    if match is None:
        return None

    # Include split to avoid accidental ID collisions
    return f"CASIA_{split}_{match.group(1).lower()}"


# Create or update CASIA video identifiers

casia_mask = features["dataset"] == "CASIA"

features.loc[
    casia_mask, "video_id"
] = features.loc[casia_mask].apply(
    lambda row: extract_casia_video_id(
        row["path"],
        row["split"]
    ),
    axis=1
)


# Validate results

casia_data = features.loc[casia_mask].copy()

missing = casia_data[
    casia_data["video_id"].isna()
]

print("=" * 50)
print("CASIA VIDEO ID VALIDATION")
print("=" * 50)

print("Total samples:", len(casia_data))
print("Missing video IDs:", len(missing))
print(
    "Unique videos:",
    casia_data["video_id"].nunique()
)

if len(missing) > 0:
    print("\nUnrecognized filenames:")
    print(
        missing["path"].head(30).to_string(
            index=False
        )
    )

    raise ValueError(
        "Video ID extraction is incomplete."
    )


# Check if a video identifier has conflicting labels

conflicting = (
    casia_data.groupby("video_id")["label"]
    .nunique()
)

assert (conflicting <= 1).all(), (
    "Some video IDs have conflicting labels. "
    "Review filename grouping."
)


# Display sample video IDs

print("\nSample video identifiers:")

print(
    casia_data[
        ["split", "label", "video_id"]
    ].head(10).to_string(index=False)
)

# Save corrected feature table

mode = "full" if FULL_RUN else "pilot"

features.to_csv(
    PIPELINE_DIR / f"all_features_{mode}.csv",
    index=False
)

print("\nVideo ID validation completed.")
print("Corrected features saved.")


In [ ]:

# ============================================================
# CELL F: TRAINING, CALIBRATION & TEST PREPARATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# 1. Convert extracted features into X and y
# ------------------------------------------------------------

def get_xy(df, cols):

    X = df[cols].to_numpy(dtype=np.float32)

    # Real = 0, Spoof = 1
    y = (
        df["label"].str.lower() == "spoof"
    ).astype(int).to_numpy()

    assert len(X) == len(y)
    assert np.isfinite(X).all()
    assert set(np.unique(y)) == {0, 1}

    return X, y


# ------------------------------------------------------------
# 2. CASIA global subject ID
# ------------------------------------------------------------
# Assumption:
# Train: local IDs 1-20
# Test:  local IDs 1-30, offset by 20
#
# Requires verification against the original
# dataset subject-mapping documentation.

def casia_global_subject(path, split):

    local_id = get_subject(path)

    if local_id is None:
        raise ValueError(
            f"Unrecognized CASIA filename: {path}"
        )

    number = int(local_id[1:])
    split = str(split).lower()

    if split == "train":

        if not 1 <= number <= 20:
            raise ValueError(
                f"Unexpected train subject: {path}"
            )

        global_number = number

    elif split == "test":

        if not 1 <= number <= 30:
            raise ValueError(
                f"Unexpected test subject: {path}"
            )

        global_number = number + 20

    else:

        raise ValueError(
            f"Unknown CASIA split: {split}"
        )

    return f"subject_{global_number:02d}"


# ------------------------------------------------------------
# 3. Limit training frames per CASIA video
# ------------------------------------------------------------

def cap_video_frames(df, cap):

    if cap is None:
        return df.copy()

    if cap <= 0:
        raise ValueError(
            "Frame cap must be positive."
        )

    if "video_id" not in df.columns:
        raise ValueError(
            "Missing video_id column. "
            "Run the Video ID Fix Cell first."
        )

    if df["video_id"].isna().any():
        raise ValueError(
            "Missing video IDs. "
            "Run and verify the Video ID Fix Cell."
        )

    pieces = []

    for _, group in df.groupby(
        ["label", "video_id"],
        dropna=False
    ):

        pieces.append(
            group.sample(
                n=min(cap, len(group)),
                random_state=SEED
            )
        )

    if not pieces:
        raise ValueError(
            "No video groups available."
        )

    return pd.concat(
        pieces,
        ignore_index=True
    )


# ------------------------------------------------------------
# 4. Prepare dataset partitions
# ------------------------------------------------------------

def splits_for(dataset):

    df = features[
        features["dataset"] == dataset
    ].copy()

    if df.empty:
        raise ValueError(
            f"No features found for {dataset}"
        )

    # --------------------------------------------------------
    # LCC-FASD: Preserve provided official splits
    # --------------------------------------------------------

    if dataset == "LCC":

        fit = df[
            df["split"] == "training"
        ].copy()

        cal = df[
            df["split"] == "development"
        ].copy()

        test = df[
            df["split"] == "evaluation"
        ].copy()

    # --------------------------------------------------------
    # CASIA-FASD
    # --------------------------------------------------------

    elif dataset == "CASIA":

        # Use corrected video identifiers
        if "video_id" not in df.columns:
            raise ValueError(
                "video_id column missing."
            )

        if df["video_id"].isna().any():
            raise ValueError(
                "CASIA video IDs are incomplete."
            )

        # Recover provisional global subject identities
        df["global_subject"] = df.apply(
            lambda row: casia_global_subject(
                row["path"],
                row["split"]
            ),
            axis=1
        )

        official_train = df[
            df["split"] == "train"
        ].copy()

        test = df[
            df["split"] == "test"
        ].copy()

        # Check subject separation
        train_subjects = set(
            official_train["global_subject"]
        )

        test_subjects = set(
            test["global_subject"]
        )

        overlap = (
            train_subjects & test_subjects
        )

        print(
            "CASIA global subject overlap:",
            sorted(overlap)
        )

        if overlap:
            raise ValueError(
                "Potential CASIA train/test "
                "subject overlap detected. "
                "Verify original subject mapping."
            )

        # Create subject-disjoint calibration split
        splitter = GroupShuffleSplit(
            n_splits=50,
            test_size=0.25,
            random_state=SEED
        )

        choice = None

        for fit_idx, cal_idx in splitter.split(
            official_train,
            groups=official_train["global_subject"]
        ):

            candidate_fit = (
                official_train.iloc[fit_idx].copy()
            )

            candidate_cal = (
                official_train.iloc[cal_idx].copy()
            )

            if (
                candidate_fit["label"].nunique() == 2
                and candidate_cal["label"].nunique() == 2
            ):

                choice = (
                    candidate_fit,
                    candidate_cal
                )

                break

        if choice is None:
            raise ValueError(
                "Unable to create a valid "
                "subject-disjoint CASIA split. "
                "Check pilot sample coverage."
            )

        fit, cal = choice

        # Check fit/calibration subject separation
        fit_subjects = set(
            fit["global_subject"]
        )

        cal_subjects = set(
            cal["global_subject"]
        )

        assert not (
            fit_subjects & cal_subjects
        ), (
            "CASIA fit/calibration "
            "subject overlap detected."
        )

        # Cap frames AFTER subject splitting
        fit = cap_video_frames(
            fit,
            TRAIN_FRAMES_PER_VIDEO
        )

    else:

        raise ValueError(
            f"Unknown dataset: {dataset}"
        )

    # --------------------------------------------------------
    # 5. Validate each partition
    # --------------------------------------------------------

    partitions = {
        "Training": fit,
        "Calibration": cal,
        "Testing": test
    }

    for name, part in partitions.items():

        if part.empty:
            raise ValueError(
                f"{dataset}: {name} is empty."
            )

        if part["label"].nunique() != 2:
            raise ValueError(
                f"{dataset}: {name} must "
                "contain both classes."
            )

        if part["path"].duplicated().any():
            raise ValueError(
                f"{dataset}: Duplicate paths "
                f"in {name}."
            )

    # Check exact file separation
    assert not (
        set(fit["path"]) & set(cal["path"])
    )

    assert not (
        set(fit["path"]) & set(test["path"])
    )

    assert not (
        set(cal["path"]) & set(test["path"])
    )

    return fit, cal, test


# ------------------------------------------------------------
# 6. Execute and display partition summary
# ------------------------------------------------------------

print("=" * 60)
print("CELL F: DATASET PREPARATION")
print("=" * 60)

for dataset in ("LCC", "CASIA"):

    fit, cal, test = splits_for(dataset)

    print("\nDataset:", dataset)

    for name, part in [
        ("Training", fit),
        ("Calibration", cal),
        ("Testing", test)
    ]:

        print(
            f"{name}: {len(part)} samples"
        )

        print(
            part["label"].value_counts().to_dict()
        )

    print("-" * 60)


print("\nCell F execution completed.")


In [ ]:

# ============================================================
# CELL G: CLASSIFIER TRAINING AND EVALUATION
# ============================================================

import time
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

SEED = 42

FFT_COLS = [
    name for name in feature_names
    if not name.startswith("dct_")
]

ALL_COLS = feature_names.copy()

assert len(FFT_COLS) == 12, (
    f"Expected 12 FFT features, got {len(FFT_COLS)}"
)

assert len(ALL_COLS) == 18, (
    f"Expected 18 combined features, got {len(ALL_COLS)}"
)

configurations = [
    ("FFT", FFT_COLS),
    ("FFT_DCT", ALL_COLS)
]

model_names = [
    "SVM",
    "RandomForest",
    "TinyMLP"
]


# ------------------------------------------------------------
# 2. Define classifiers
# ------------------------------------------------------------

def make_model(name):

    if name == "SVM":

        return make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=1.0,
                probability=True,
                random_state=SEED
            )
        )

    elif name == "RandomForest":

        return RandomForestClassifier(
            n_estimators=100,
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        )

    elif name == "TinyMLP":

        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(16, 8),
                activation="relu",
                solver="adam",
                max_iter=150,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=10,
                random_state=SEED
            )
        )

    else:

        raise ValueError(
            f"Unknown classifier: {name}"
        )


# ------------------------------------------------------------
# 3. Prepare experiment output
# ------------------------------------------------------------

PIPELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

experiment_rows = []

mode = (
    "full" if FULL_RUN
    else "pilot"
)

print("=" * 65)
print("CELL G: CLASSIFIER TRAINING")
print("Run mode:", mode.upper())
print("=" * 65)


# ------------------------------------------------------------
# 4. Train and evaluate each dataset
# ------------------------------------------------------------

for dataset in ("LCC", "CASIA"):

    print("\n" + "=" * 65)
    print("DATASET:", dataset)
    print("=" * 65)

    # Reuse Cell F
    fit, cal, test = splits_for(dataset)

    for feature_set, columns in configurations:

        print("\nFeature configuration:", feature_set)

        # Convert DataFrames to numerical arrays
        Xfit, yfit = get_xy(fit, columns)
        Xcal, ycal = get_xy(cal, columns)
        Xtest, ytest = get_xy(test, columns)

        # Train three lightweight classifiers
        for model_name in model_names:

            print("\nTraining:", model_name)

            model = make_model(model_name)

            Xtrain = Xfit
            ytrain = yfit

            # Restrict SVM training size
            # for computational efficiency
            if (
                model_name == "SVM"
                and len(Xfit) > SVM_TRAIN_CAP
            ):

                indices = np.arange(len(Xfit))

                selected, _ = train_test_split(
                    indices,
                    train_size=SVM_TRAIN_CAP,
                    stratify=yfit,
                    random_state=SEED
                )

                Xtrain = Xfit[selected]
                ytrain = yfit[selected]

            # --------------------------------------------
            # Train classifier
            # --------------------------------------------

            start_train = time.perf_counter()

            model.fit(
                Xtrain,
                ytrain
            )

            train_seconds = (
                time.perf_counter() - start_train
            )

            # --------------------------------------------
            # Calibration: threshold selection
            # --------------------------------------------

            calibration_scores = (
                model.predict_proba(Xcal)[:, 1]
            )

            threshold = choose_threshold(
                calibration_scores,
                ycal
            )

            # --------------------------------------------
            # Evaluate held-out test samples
            # --------------------------------------------

            start_test = time.perf_counter()

            test_scores = (
                model.predict_proba(Xtest)[:, 1]
            )

            inference_seconds = (
                time.perf_counter() - start_test
            )

            results = metrics(
                ytest,
                test_scores,
                threshold
            )

            # --------------------------------------------
            # Save experiment results
            # --------------------------------------------

            record = {

                "dataset": dataset,

                "features": feature_set,

                "model": model_name,

                "run_mode": mode,

                "trained_n": int(len(ytrain)),

                "cal_n": int(len(ycal)),

                "test_n": int(len(ytest)),

                "train_seconds": float(
                    train_seconds
                ),

                "inference_seconds": float(
                    inference_seconds
                ),

                "ms_per_frame": float(
                    1000 * inference_seconds
                    / len(ytest)
                ),

                **results
            }

            experiment_rows.append(record)

            # --------------------------------------------
            # Save trained model and metadata
            # --------------------------------------------

            filename = (
                f"{dataset}_{feature_set}_{model_name}"
            ).lower()

            model_path = (
                PIPELINE_DIR
                / f"{filename}_model.joblib"
            )

            joblib.dump(

                {
                    "model": model,

                    "columns": columns,

                    "threshold": threshold,

                    "image_size": IMG_SIZE,

                    "training_samples": int(
                        len(ytrain)
                    ),

                    "run_mode": mode

                },

                model_path
            )

            print(
                "Training time:",
                round(train_seconds, 3),
                "seconds"
            )

            print(
                "Accuracy:",
                round(results["accuracy"], 4)
            )

            print(
                "APCER:",
                round(results["APCER"], 4)
            )

            print(
                "BPCER:",
                round(results["BPCER"], 4)
            )

            print(
                "ACER:",
                round(results["ACER"], 4)
            )

            print(
                "ROC-AUC:",
                round(results["ROC_AUC"], 4)
            )

            print(
                "Model saved:",
                model_path
            )


# ------------------------------------------------------------
# 5. Create comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame(
    experiment_rows
)

comparison_columns = [

    "dataset",
    "features",
    "model",
    "run_mode",
    "trained_n",
    "cal_n",
    "test_n",
    "accuracy",
    "APCER",
    "BPCER",
    "ACER",
    "ROC_AUC",
    "PR_AUC",
    "train_seconds",
    "ms_per_frame"

]

print("\n" + "=" * 65)
print("FINAL EXPERIMENT COMPARISON")
print("=" * 65)

display(
    comparison[
        comparison_columns
    ].round(4)
)


# ------------------------------------------------------------
# 6. Save comparison results
# ------------------------------------------------------------

comparison_path = (
    PIPELINE_DIR
    / f"comparison_{mode}.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

assert len(comparison) == 12, (
    "Expected 12 completed experiments."
)

print("\nTotal experiments:", len(comparison))

print(
    "Comparison CSV saved:",
    comparison_path
)

print("\nCell G execution completed.")


In [ ]:

# ============================================================
# CELL H: PERFORMANCE VISUALIZATION AND EFFICIENCY
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# 1. Load experiment results
# ------------------------------------------------------------

mode = "full" if FULL_RUN else "pilot"

comparison_path = (
    PIPELINE_DIR / f"comparison_{mode}.csv"
)

if not comparison_path.exists():
    raise FileNotFoundError(
        "Cell G results not found. "
        "Run Cell G successfully first."
    )

comparison = pd.read_csv(comparison_path)

assert len(comparison) == 12, (
    "Expected 12 experiment results."
)

required_columns = [
    "dataset",
    "features",
    "model",
    "accuracy",
    "APCER",
    "BPCER",
    "ACER",
    "ROC_AUC",
    "train_seconds",
    "ms_per_frame"
]

assert all(
    col in comparison.columns
    for col in required_columns
)

GRAPH_DIR = PIPELINE_DIR / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Loaded experiments:", len(comparison))


# ------------------------------------------------------------
# 2. Create descriptive experiment labels
# ------------------------------------------------------------

comparison["experiment"] = (
    comparison["dataset"]
    + " | "
    + comparison["features"]
    + " | "
    + comparison["model"]
)


# ------------------------------------------------------------
# 3. ACER comparison
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    comparison["experiment"],
    comparison["ACER"]
)

ax.set_xlabel("ACER")
ax.set_ylabel("Experiment")
ax.set_title(
    f"Face Anti-Spoofing: ACER Comparison ({mode})"
)

ax.set_xlim(
    0,
    max(0.05, comparison["ACER"].max() * 1.15)
)

ax.invert_yaxis()

plt.tight_layout()

plt.savefig(
    GRAPH_DIR / f"acer_comparison_{mode}.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 4. Accuracy comparison
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    comparison["experiment"],
    comparison["accuracy"]
)

ax.set_xlabel("Accuracy")
ax.set_ylabel("Experiment")

ax.set_title(
    f"Classification Accuracy ({mode})"
)

ax.set_xlim(0, 1)

ax.invert_yaxis()

plt.tight_layout()

plt.savefig(
    GRAPH_DIR / f"accuracy_comparison_{mode}.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 5. APCER vs BPCER
# ------------------------------------------------------------

positions = np.arange(len(comparison))
bar_width = 0.38

fig, ax = plt.subplots(figsize=(13, 7))

ax.bar(
    positions - bar_width / 2,
    comparison["APCER"],
    width=bar_width,
    label="APCER"
)

ax.bar(
    positions + bar_width / 2,
    comparison["BPCER"],
    width=bar_width,
    label="BPCER"
)

ax.set_xticks(positions)

ax.set_xticklabels(
    comparison["experiment"],
    rotation=75,
    ha="right"
)

ax.set_ylabel("Error Rate")
ax.set_title(
    f"APCER and BPCER Comparison ({mode})"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    GRAPH_DIR / f"apcer_bpcer_{mode}.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 6. Classifier training-time comparison
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    comparison["experiment"],
    comparison["train_seconds"]
)

ax.set_xlabel("Training Time (seconds)")
ax.set_title(
    f"Classifier Training Time ({mode})"
)

ax.invert_yaxis()

plt.tight_layout()

plt.savefig(
    GRAPH_DIR / f"training_time_{mode}.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 7. Model-only inference latency
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    comparison["experiment"],
    comparison["ms_per_frame"]
)

ax.set_xlabel(
    "Model-only Inference Time (ms/sample)"
)

ax.set_title(
    f"Classifier Inference Latency ({mode})"
)

ax.invert_yaxis()

plt.tight_layout()

plt.savefig(
    GRAPH_DIR / f"inference_latency_{mode}.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8. Measure preprocessing + feature extraction
# ------------------------------------------------------------

# This is a small exploratory timing test.
# It is not a definitive hardware benchmark.

TIMING_SAMPLES = 12

timing_results = []

for dataset, group in phase_manifest.groupby(
    "dataset"
):

    sample_group = group.sample(
        n=min(TIMING_SAMPLES, len(group)),
        random_state=SEED
    )

    print(
        "\nTiming dataset:",
        dataset
    )

    for row in sample_group.itertuples(index=False):

        start = time.perf_counter()

        gray, status = read_preprocessed(
            row.path,
            row.dataset
        )

        if gray is None:

            timing_results.append({
                "dataset": dataset,
                "path": row.path,
                "success": False,
                "seconds": np.nan,
                "reason": status
            })

            continue

        extracted = all_features(gray)

        elapsed = (
            time.perf_counter() - start
        )

        timing_results.append({
            "dataset": dataset,
            "path": row.path,
            "success": extracted is not None,
            "seconds": elapsed,
            "reason": (
                "ok"
                if extracted is not None
                else "invalid_features"
            )
        })


# ------------------------------------------------------------
# 9. Analyze timing results
# ------------------------------------------------------------

timing_df = pd.DataFrame(
    timing_results
)

timing_path = (
    PIPELINE_DIR
    / f"pipeline_timing_{mode}.csv"
)

timing_df.to_csv(
    timing_path,
    index=False
)

successful_timing = timing_df[
    timing_df["success"]
].copy()

if successful_timing.empty:
    raise ValueError(
        "No successful timing measurements."
    )

timing_summary = (
    successful_timing
    .groupby("dataset")["seconds"]
    .agg(
        ["count", "mean", "median", "std"]
    )
)

timing_summary["mean_ms"] = (
    timing_summary["mean"] * 1000
)

timing_summary["median_ms"] = (
    timing_summary["median"] * 1000
)

print("\n" + "=" * 60)
print("PREPROCESSING + FEATURE EXTRACTION TIMING")
print("=" * 60)

display(
    timing_summary.round(4)
)

timing_summary.to_csv(
    PIPELINE_DIR
    / f"pipeline_timing_summary_{mode}.csv"
)


# ------------------------------------------------------------
# 10. Save consolidated efficiency report
# ------------------------------------------------------------

efficiency_columns = [
    "dataset",
    "features",
    "model",
    "trained_n",
    "test_n",
    "train_seconds",
    "inference_seconds",
    "ms_per_frame"
]

efficiency_report = comparison[
    efficiency_columns
].copy()

efficiency_path = (
    PIPELINE_DIR
    / f"efficiency_report_{mode}.csv"
)

efficiency_report.to_csv(
    efficiency_path,
    index=False
)


# ------------------------------------------------------------
# 11. Final output summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CELL H COMPLETED")
print("=" * 60)

print("Graphs directory:", GRAPH_DIR)

print(
    "Saved visualization files:",
    len(list(GRAPH_DIR.glob(f"*_{mode}.png")))
)

print("Timing report:", timing_path)

print(
    "Efficiency report:",
    efficiency_path
)

print(
    "\nNOTE: Pilot results are exploratory."
)

print(
    "Inference latency excludes preprocessing "
    "and feature extraction."
)


In [ ]:

# ============================================================
# CELL I: CROSS-DATASET EVALUATION
# ============================================================

import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

CROSS_DATASET = True

mode = "full" if FULL_RUN else "pilot"

configurations = [
    ("FFT", FFT_COLS),
    ("FFT_DCT", ALL_COLS)
]

model_names = [
    "SVM",
    "RandomForest",
    "TinyMLP"
]

experiments = [
    ("LCC", "CASIA"),
    ("CASIA", "LCC")
]

cross_rows = []

PIPELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Check previous training results
# ------------------------------------------------------------

comparison_path = (
    PIPELINE_DIR / f"comparison_{mode}.csv"
)

if not comparison_path.exists():

    raise FileNotFoundError(
        "Cell G comparison results are missing. "
        "Run Cell G first."
    )

previous_results = pd.read_csv(
    comparison_path
)

assert len(previous_results) == 12, (
    "Expected 12 trained experiment results."
)

print("=" * 65)
print("CELL I: CROSS-DATASET EVALUATION")
print("Run mode:", mode.upper())
print("=" * 65)


# ------------------------------------------------------------
# 3. Perform cross-dataset experiments
# ------------------------------------------------------------

if CROSS_DATASET:

    for source, target in experiments:

        print("\n" + "=" * 65)

        print(
            "Training source:",
            source
        )

        print(
            "Testing target:",
            target
        )

        print("=" * 65)

        # Load target dataset's held-out test partition
        _, _, target_test = splits_for(
            target
        )

        for feature_set, columns in configurations:

            Xtarget, ytarget = get_xy(
                target_test,
                columns
            )

            for model_name in model_names:

                # Construct saved model filename
                filename = (
                    f"{source}_{feature_set}_{model_name}"
                ).lower()

                model_path = (
                    PIPELINE_DIR
                    / f"{filename}_model.joblib"
                )

                if not model_path.exists():

                    raise FileNotFoundError(
                        f"Missing model: {model_path}"
                    )

                # Load previously trained model
                package = joblib.load(
                    model_path
                )

                model = package["model"]

                saved_columns = package["columns"]

                threshold = package["threshold"]

                # Verify feature compatibility
                assert saved_columns == columns, (
                    "Feature-column mismatch."
                )

                assert package["image_size"] == IMG_SIZE, (
                    "Image-size mismatch."
                )

                assert package["run_mode"] == mode, (
                    "Model run-mode mismatch."
                )

                # ----------------------------------------
                # Predict on the other dataset
                # ----------------------------------------

                start = time.perf_counter()

                target_scores = (
                    model.predict_proba(
                        Xtarget
                    )[:, 1]
                )

                inference_seconds = (
                    time.perf_counter() - start
                )

                # ----------------------------------------
                # Evaluate using SOURCE threshold
                # ----------------------------------------
                # Important:
                # Do not recalibrate the threshold
                # using the target test dataset.

                results = metrics(
                    ytarget,
                    target_scores,
                    threshold
                )

                # ----------------------------------------
                # Save results
                # ----------------------------------------

                record = {

                    "source_dataset": source,

                    "target_dataset": target,

                    "features": feature_set,

                    "model": model_name,

                    "run_mode": mode,

                    "test_samples": int(
                        len(ytarget)
                    ),

                    "inference_seconds": float(
                        inference_seconds
                    ),

                    "ms_per_frame": float(
                        1000 * inference_seconds
                        / len(ytarget)
                    ),

                    **results
                }

                cross_rows.append(
                    record
                )

                print(
                    source,
                    "->",
                    target,
                    "|",
                    feature_set,
                    "|",
                    model_name
                )

                print(
                    "Accuracy:",
                    round(results["accuracy"], 4)
                )

                print(
                    "APCER:",
                    round(results["APCER"], 4)
                )

                print(
                    "BPCER:",
                    round(results["BPCER"], 4)
                )

                print(
                    "ACER:",
                    round(results["ACER"], 4)
                )

                print(
                    "ROC-AUC:",
                    round(results["ROC_AUC"], 4)
                )

                print("-" * 55)


# ------------------------------------------------------------
# 4. Create cross-dataset comparison table
# ------------------------------------------------------------

cross = pd.DataFrame(
    cross_rows
)

assert len(cross) == 12, (
    "Expected 12 cross-dataset experiments."
)

display_columns = [

    "source_dataset",

    "target_dataset",

    "features",

    "model",

    "test_samples",

    "accuracy",

    "APCER",

    "BPCER",

    "ACER",

    "ROC_AUC",

    "PR_AUC",

    "ms_per_frame"

]


print("\n" + "=" * 65)

print("CROSS-DATASET RESULTS")

print("=" * 65)

display(
    cross[
        display_columns
    ].round(4)
)


# ------------------------------------------------------------
# 5. Export cross-dataset results
# ------------------------------------------------------------

cross_path = (
    PIPELINE_DIR
    / f"cross_dataset_{mode}.csv"
)

cross.to_csv(
    cross_path,
    index=False
)

print("\nCross-dataset CSV saved:")
print(cross_path)


# ------------------------------------------------------------
# 6. Create comparison graph
# ------------------------------------------------------------

import matplotlib.pyplot as plt

cross["experiment"] = (
    cross["source_dataset"]
    + " -> "
    + cross["target_dataset"]
    + " | "
    + cross["features"]
    + " | "
    + cross["model"]
)

fig, ax = plt.subplots(
    figsize=(13, 8)
)

ax.barh(
    cross["experiment"],
    cross["ACER"]
)

ax.set_xlabel(
    "Sample/Frame-Level ACER"
)

ax.set_title(
    f"Cross-Dataset ACER Comparison ({mode})"
)

ax.invert_yaxis()

plt.tight_layout()

graph_dir = (
    PIPELINE_DIR / "graphs"
)

graph_dir.mkdir(
    parents=True,
    exist_ok=True
)

graph_path = (
    graph_dir
    / f"cross_dataset_acer_{mode}.png"
)

plt.savefig(
    graph_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)

print("CELL I EXECUTION COMPLETED")

print("=" * 65)

print(
    "Total cross-dataset experiments:",
    len(cross)
)

print(
    "Results CSV:",
    cross_path
)

print(
    "Comparison graph:",
    graph_path
)

print(
    "\nIMPORTANT: Pilot results are exploratory."
)

print(
    "Dataset preprocessing differences and "
    "subject-mapping assumptions must be "
    "reviewed before publication."
)


In [ ]:
import json

cas = features[
    features["dataset"] == "CASIA"
].copy()

cas["global_subject"] = cas.apply(
    lambda r: casia_global_subject(
        r["path"],
        r["split"]
    ),
    axis=1
)

train_ids = set(
    cas.loc[
        cas["split"] == "train",
        "global_subject"
    ]
)

test_ids = set(
    cas.loc[
        cas["split"] == "test",
        "global_subject"
    ]
)

overlap = sorted(train_ids & test_ids)

print("CASIA global subject overlap:", overlap)

assert not overlap, (
    "Potential subject overlap detected."
)

audit = {
    "protocol": "Provisional CASIA subject mapping",
    "assumption": (
        "Kaggle filenames preserve original "
        "CASIA-FASD local subject numbering"
    ),
    "global_subject_overlap": overlap,
    "mapping_verified_against_original_files": False,
    "pilot": not FULL_RUN,
    "lcc_subject_overlap": "NOT_VERIFIED"
}

audit_path = (
    PIPELINE_DIR / "protocol_audit_corrected.json"
)

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print("Audit file created:", audit_path)

In [ ]:

# ============================================================
# CELL J: FINAL RESULTS VALIDATION AND EXPORT
# ============================================================

import os
import json
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import sklearn


# ------------------------------------------------------------
# 1. Configure output
# ------------------------------------------------------------

mode = "full" if FULL_RUN else "pilot"

PIPELINE_DIR = Path(PIPELINE_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("CELL J: FINAL RESEARCH EXPORT")
print("Mode:", mode.upper())
print("=" * 65)


# ------------------------------------------------------------
# 2. Check mandatory experiment results
# ------------------------------------------------------------

required_files = [
    f"all_features_{mode}.csv",
    f"failures_{mode}.csv",
    f"comparison_{mode}.csv",
    f"cross_dataset_{mode}.csv",
    f"efficiency_report_{mode}.csv",
    f"pipeline_timing_{mode}.csv",
    "protocol_audit_corrected.json"
]

missing_files = [
    filename
    for filename in required_files
    if not (PIPELINE_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print("\nAll mandatory result files found.")


# ------------------------------------------------------------
# 3. Validate experiment results
# ------------------------------------------------------------

comparison = pd.read_csv(
    PIPELINE_DIR / f"comparison_{mode}.csv"
)

cross = pd.read_csv(
    PIPELINE_DIR / f"cross_dataset_{mode}.csv"
)

assert len(comparison) == 12, (
    "Expected 12 within-dataset experiments."
)

assert len(cross) == 12, (
    "Expected 12 cross-dataset experiments."
)

for df in (comparison, cross):

    for column in (
        "accuracy",
        "APCER",
        "BPCER",
        "ACER",
        "ROC_AUC"
    ):
        assert df[column].notna().all(), (
            f"Missing values in {column}"
        )

        assert df[column].between(0, 1).all(), (
            f"Invalid metric values: {column}"
        )

    calculated_acer = (
        df["APCER"] + df["BPCER"]
    ) / 2

    assert np.allclose(
        df["ACER"],
        calculated_acer
    ), "ACER calculation mismatch"

print("Experiment result validation passed.")


# ------------------------------------------------------------
# 4. Verify trained model files
# ------------------------------------------------------------

model_files = list(
    PIPELINE_DIR.glob("*_model.joblib")
)

expected_models = [
    f"{dataset}_{feature_set}_{model}_model.joblib".lower()
    for dataset in ("LCC", "CASIA")
    for feature_set in ("FFT", "FFT_DCT")
    for model in ("SVM", "RandomForest", "TinyMLP")
]

missing_models = [
    name
    for name in expected_models
    if not (PIPELINE_DIR / name).exists()
]

if missing_models:
    raise FileNotFoundError(
        "Missing trained models:\n"
        + "\n".join(missing_models)
    )

print(
    "Required trained models:",
    len(expected_models)
)


# ------------------------------------------------------------
# 5. Load protocol audit
# ------------------------------------------------------------

with open(
    PIPELINE_DIR / "protocol_audit_corrected.json"
) as f:

    protocol_audit = json.load(f)

if protocol_audit.get("global_subject_overlap"):
    raise ValueError(
        "Protocol audit reports potential "
        "subject overlap. Review before export."
    )

print("Protocol audit file loaded.")

print(
    "IMPORTANT: Original CASIA subject mapping "
    "still requires independent verification."
)


# ------------------------------------------------------------
# 6. Prepare research metadata
# ------------------------------------------------------------

metadata = {

    "project": (
        "Lightweight Face Anti-Spoofing "
        "using Frequency Domain Analysis"
    ),

    "experiment_mode": mode,

    "results_status": (
        "exploratory_pilot"
        if not FULL_RUN
        else "full_run_protocol_verification_pending"
    ),

    "datasets": [
        "LCC-FASD",
        "CASIA-FASD"
    ],

    "features": [
        "FFT",
        "FFT+DCT"
    ],

    "classifiers": [
        "SVM",
        "RandomForest",
        "TinyMLP"
    ],

    "random_seed": SEED,

    "image_resolution": [
        IMG_SIZE,
        IMG_SIZE
    ],

    "dct_block_size": DCT_SIZE,

    "casia_training_frames_per_video":
        TRAIN_FRAMES_PER_VIDEO,

    "svm_training_cap":
        SVM_TRAIN_CAP,

    "within_dataset_experiments":
        len(comparison),

    "cross_dataset_experiments":
        len(cross),

    "saved_model_count":
        len(expected_models),

    "environment": {

        "python": platform.python_version(),

        "numpy": np.__version__,

        "pandas": pd.__version__,

        "opencv": cv2.__version__,

        "scikit_learn": sklearn.__version__
    },

    "research_limitations": [

        "CASIA subject mapping requires "
        "verification against original metadata.",

        "LCC subject-level leakage audit "
        "has not been completed.",

        "Full-dataset results require "
        "protocol validation.",

        "Preprocessing differences may affect "
        "cross-dataset performance.",

        "Current APCER and BPCER use pooled "
        "sample/frame-level calculations.",

        "Model inference timing does not "
        "include image preprocessing.",

        "Additional statistical evaluation "
        "is required for publication."
    ]
}


# ------------------------------------------------------------
# 7. Save research metadata
# ------------------------------------------------------------

metadata_path = (
    PIPELINE_DIR / f"research_metadata_{mode}.json"
)

with open(metadata_path, "w") as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print("Research metadata saved.")


# ------------------------------------------------------------
# 8. Create file inventory
# ------------------------------------------------------------

inventory = []

for path in PIPELINE_DIR.rglob("*"):

    if path.is_file():

        inventory.append({

            "file": str(
                path.relative_to(PIPELINE_DIR)
            ),

            "size_bytes": path.stat().st_size,

            "extension": path.suffix
        })


inventory_df = pd.DataFrame(inventory)

inventory_path = (
    PIPELINE_DIR / f"file_inventory_{mode}.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

print(
    "Files inventoried:",
    len(inventory_df)
)


# ------------------------------------------------------------
# 9. Create final research archive
# ------------------------------------------------------------

archive_path = (
    OUTPUT_DIR
    / f"face_antispoofing_research_{mode}.zip"
)

with zipfile.ZipFile(
    archive_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in PIPELINE_DIR.rglob("*"):

        if path.is_file():

            archive.write(
                path,
                arcname=str(
                    path.relative_to(PIPELINE_DIR)
                )
            )


# ------------------------------------------------------------
# 10. Verify ZIP integrity
# ------------------------------------------------------------

with zipfile.ZipFile(
    archive_path,
    "r"
) as archive:

    corrupted_file = archive.testzip()

    if corrupted_file is not None:

        raise RuntimeError(
            "ZIP integrity check failed: "
            + corrupted_file
        )

    archived_files = archive.namelist()


archive_size_mb = (
    archive_path.stat().st_size
    / (1024 * 1024)
)


# ------------------------------------------------------------
# 11. Final report
# ------------------------------------------------------------

print("\n" + "=" * 65)

print("FINAL EXPORT SUMMARY")

print("=" * 65)

print(
    "Within-dataset experiments:",
    len(comparison)
)

print(
    "Cross-dataset experiments:",
    len(cross)
)

print(
    "Required trained models:",
    len(expected_models)
)

print(
    "Files inside archive:",
    len(archived_files)
)

print(
    "Archive size:",
    round(archive_size_mb, 2),
    "MB"
)

print(
    "Archive saved:",
    archive_path
)

print("\nZIP integrity verification passed.")

print("\nCELL J EXECUTION COMPLETED.")

if not FULL_RUN:

    print(
        "\nWARNING: These are PILOT results. "
        "Do not present them as final research findings."
    )

else:

    print(
        "\nFull execution completed. "
        "Dataset protocols and results "
        "still require scientific validation."
    )
